In [28]:
from pathlib import Path
import shutil

import geopandas as gpd
import pandas as pd

from satchip import models, download_data, merge_modality, generate_labels, chip_data, view

In [2]:
shp_path = 'hwds_pristine'
df = gpd.read_file(shp_path)

df['SwathDate'] = pd.to_datetime(df['SwathDate'], format='%Y-%m-%d')
df['HLSDate'] = pd.to_datetime(df['HLSDate'], format='%Y-%m-%d')

#df = df[df['Visibility'] == '1']

In [25]:
DATA_PATH = Path.cwd() / 'data'

def make_mod_paths(modality):
    base_path = DATA_PATH / modality['id']

    return {
        'modality': modality,
        'raw': base_path / 'raw',
        'merged': base_path / 'merged',
        'wgs84': base_path / 'wgs84',
        'stacked': base_path / 'stacked',
        'warped': base_path / 'warped',
        'chips': base_path / 'chips',
        'plots': base_path / 'plots'
    }

mod_paths = {
    modality['id']: make_mod_paths(modality) for modality in (models.HLS_S30, models.HLS_L30)
}
mod_paths

{'HLS_S30': {'modality': {'id': 'HLS_S30',
   'collection': 'HLSS30',
   'bands': (Band(id='B02', name='Blue', shortname='B'),
    Band(id='B03', name='Green', shortname='G'),
    Band(id='B04', name='Red', shortname='R'),
    Band(id='B8A', name='NIR Narrow', shortname='N'),
    Band(id='B11', name='SWIR 1', shortname='SW1'),
    Band(id='B12', name='SWIR 2', shortname='SW2'),
    Band(id='Fmask', name='Cloud Mask', shortname='Fmask'))},
  'raw': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/raw'),
  'merged': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged'),
  'wgs84': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/wgs84'),
  'stacked': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/stacked'),
  'warped': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/warped'),
  'chips': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/chips'),
  'plot

In [ ]:
event_files = []

for paths in mod_paths.values():
    print(paths['modality'], paths['raw'])

    for idx, row in df.iterrows():
        modality = paths['modality']

        damage_event = models.Event(
            name=row['HLSID'], 
            date=row['HLSDate'], 
            wgs84_geometry=row['geometry'], 
            buffer_m=10000
        )

        local = download_data.download_data(damage_event, modality, paths['raw'])

        if not local:
            print(f'no {modality["id"]} data for {damage_event.name}')
            continue
        else:
            print(f'Found {modality["id"]} data for {damage_event.name}')

        merged_event = merge_modality.merge_modality(
            local, 
            modality, 
            event=damage_event, 
            output_path=paths['merged']
        )

        stacked_filename = paths['stacked'] / f'{damage_event.name}.{modality["id"]}.stacked.tif'
        data_bands, fmask = merged_event[:-1], merged_event[-1]

        stacked = merge_modality.stack_bands(data_bands, stacked_filename)

        label = generate_labels.binary_mask_from_template(stacked, damage_event, paths['wgs84'])
        mask, bands, fmask = merge_modality.warp_to_reference(
            reference_path=label,
            data_files=[stacked, fmask],
            output_dir=paths['warped'],
            bounding_box_wgs84=damage_event.buffered_geometry().bounds,
        )
        print(f'Adding event files for {damage_event.name}')
        event_files.append((damage_event, modality, (mask, bands, fmask)))

        view.view_merged(
            bands, 
            damage_event, 
            modality, 
            rgb_bands=[2, 1, 0],
            quite=True, 
            save_to_file=paths['plots'] / f'{damage_event.name}.{modality["id"]}.merged.plot.png'
        )

{'id': 'HLS_S30', 'collection': 'HLSS30', 'bands': (Band(id='B02', name='Blue', shortname='B'), Band(id='B03', name='Green', shortname='G'), Band(id='B04', name='Red', shortname='R'), Band(id='B8A', name='NIR Narrow', shortname='N'), Band(id='B11', name='SWIR 1', shortname='SW1'), Band(id='B12', name='SWIR 2', shortname='SW2'), Band(id='Fmask', name='Cloud Mask', shortname='Fmask'))} /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/raw


QUEUEING TASKS | : 100%|██████████| 18/18 [00:00<00:00, 11050.57it/s]
PROCESSING TASKS | : 100%|██████████| 18/18 [00:00<00:00, 280659.75it/s]
COLLECTING RESULTS | : 100%|██████████| 18/18 [00:00<00:00, 397355.12it/s]


Found HLS_S30 data for 95a
generated: /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/wgs84/95a.MASK.tif
Adding event files for 95a


QUEUEING TASKS | : 100%|██████████| 36/36 [00:00<00:00, 11599.83it/s]
PROCESSING TASKS | : 100%|██████████| 36/36 [00:00<00:00, 461758.24it/s]
COLLECTING RESULTS | : 100%|██████████| 36/36 [00:00<00:00, 766471.80it/s]


Found HLS_S30 data for 102a
generated: /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/wgs84/102a.MASK.tif
Adding event files for 102a


...

Found HLS_L30 data for 1373d
generated: /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/wgs84/1373d.MASK.tif
Adding event files for 1373d


QUEUEING TASKS | : 100%|██████████| 30/30 [00:00<00:00, 12795.31it/s]
PROCESSING TASKS | : 100%|██████████| 30/30 [00:00<00:00, 455902.61it/s]
COLLECTING RESULTS | : 100%|██████████| 30/30 [00:00<00:00, 619847.88it/s]


Found HLS_L30 data for 1373e
generated: /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/wgs84/1373e.MASK.tif
Adding event files for 1373e


QUEUEING TASKS | : 100%|██████████| 15/15 [00:00<00:00, 10602.39it/s]
PROCESSING TASKS | : 100%|██████████| 15/15 [00:00<00:00, 266587.12it/s]
COLLECTING RESULTS | : 100%|██████████| 15/15 [00:00<00:00, 473041.80it/s]


Found HLS_L30 data for 1376a
generated: /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/wgs84/1376a.MASK.tif
Adding event files for 1376a


Remove any bad aquisitions from `data/{modality}/plots` by remove the .png 

In [ ]:
def backup_plots():
    for paths in mod_paths.values():
        plots = paths['plots']
        plots_backup = plots.parent / 'plots-backup'
        shutil.copytree(plots, plots_backup, dirs_exist_ok=True)

# backup_plots()

In [56]:
def restore_plots():
    for paths in mod_paths.values():
        plots = paths['plots']
        plots_backup = plots.parent / 'plots-backup'
        shutil.copytree(plots_backup, plots, dirs_exist_ok=True)

restore_plots()

In [57]:
keepers = {}

for paths in mod_paths.values():
    modality = paths['modality']
    mod_keepers = [plot.name.split('.')[0] for plot in paths['plots'].glob('*.png')]
    keepers[modality['id']] = mod_keepers

keepers

{'HLS_S30': ['628a',
  '1079a',
  '597b',
  '1069a',
  '1380a',
  '116d',
  '648a',
  '129a',
  '126b',
  '638a',
  '116e',
  '1338a',
  '1347a',
  '102a',
  '106d',
  '1373e',
  '623d',
  '892a',
  '1373d',
  '1064a',
  '889b',
  '134e',
  '1055a',
  '889a',
  '95a',
  '126a',
  '134a',
  '130a',
  '623a',
  '1373a',
  '116a'],
 'HLS_L30': ['116a',
  '106d',
  '126b',
  '1344a',
  '1378a',
  '1338a',
  '1376a',
  '133a',
  '613b',
  '915a',
  '1052a',
  '623d',
  '628a',
  '889a',
  '130a',
  '126a',
  '106a',
  '623a',
  '129a']}

In [58]:
print(event_files[0])
evt, modality, (m, b, f) = event_files[0]

(Event(name='95a', date=Timestamp('2018-07-02 00:00:00'), wgs84_geometry=<POLYGON Z ((-97.463 41.726 0, -97.457 41.728 0, -97.453 41.736 0, -97.447 4...>, buffer_m=10000), {'id': 'HLS_S30', 'collection': 'HLSS30', 'bands': (Band(id='B02', name='Blue', shortname='B'), Band(id='B03', name='Green', shortname='G'), Band(id='B04', name='Red', shortname='R'), Band(id='B8A', name='NIR Narrow', shortname='N'), Band(id='B11', name='SWIR 1', shortname='SW1'), Band(id='B12', name='SWIR 2', shortname='SW2'), Band(id='Fmask', name='Cloud Mask', shortname='Fmask'))}, (PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/warped/95a.MASK.tif'), PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/warped/95a.HLS_S30.stacked.tif'), PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/warped/95a.HLS_S30.2018-07-02.Fmask.tif')))


In [59]:
all_base = DATA_PATH / 'chips'
output_base = DATA_PATH / 'output'

chip_paths = {
    'all': {
        'label': all_base / 'LABEL',
        'hls': all_base / 'HLS',
        'other': all_base / 'OTHER',
        'plots': all_base / 'PLOTS',
    },
    'output': {
        'label': output_base / 'LABEL',
        'hls': output_base / 'HLS',
        'other': output_base / 'OTHER',
        'plots': all_base / 'PLOTS',
    }
}

In [60]:
chipped_events = set() 
filtered_chips = []
all_chips = []

for evt, modality, (m, b, f) in event_files:
    if evt.name not in keepers[modality['id']]:
        continue

    print('Chipping: ', evt.name, m.name, b.name, f.name)
    if evt.name in chipped_events:
        print(f'Already chipped event {evt.name}: skipping...')
        continue

    chipped_events.add(evt.name)
    grid = chip_data.make_grid_from_reference(m)

    label_chips = chip_data.chip_data(grid, m, chip_paths['all']['label'])
    data_chips = chip_data.chip_data(grid, b, chip_paths['all']['hls'])
    fmask_chips = chip_data.chip_data(grid, f, chip_paths['all']['other'])

    stacks = chip_data.make_chip_stacks(data_chips, fmask_chips, label_chips, modality)
    all_chips += stacks
    view.view_chips(
        stacks, 
        modality, 
        rgb_bands=[2, 1, 0], 
        save_to_file=chip_paths['all']['plots'] / f'{evt.name}.chips.png', 
        quite=True
    )
    
    filtered = chip_data.filter_chips(stacks)

    if len(filtered) == 0:
        print(f'Filtered out all chips for {evt.name}')
        continue

    filtered_chips += filtered
    view.view_chips(
        filtered, 
        modality, 
        rgb_bands=[2, 1, 0], 
        save_to_file=chip_paths['output']['plots'] / f'{evt.name}.filtered.png', 
        quite=True
    )

Chipping:  95a 95a.MASK.tif 95a.HLS_S30.stacked.tif 95a.HLS_S30.2018-07-02.Fmask.tif
Chipping:  102a 102a.MASK.tif 102a.HLS_S30.stacked.tif 102a.HLS_S30.2019-07-17.Fmask.tif
Chipping:  106d 106d.MASK.tif 106d.HLS_S30.stacked.tif 106d.HLS_S30.2019-08-28.Fmask.tif
Chipping:  126a 126a.MASK.tif 126a.HLS_S30.stacked.tif 126a.HLS_S30.2018-08-11.Fmask.tif
Chipping:  126b 126b.MASK.tif 126b.HLS_S30.stacked.tif 126b.HLS_S30.2018-08-11.Fmask.tif
Chipping:  129a 129a.MASK.tif 129a.HLS_S30.stacked.tif 129a.HLS_S30.2018-08-07.Fmask.tif
Chipping:  130a 130a.MASK.tif 130a.HLS_S30.stacked.tif 130a.HLS_S30.2018-08-07.Fmask.tif
Chipping:  134a 134a.MASK.tif 134a.HLS_S30.stacked.tif 134a.HLS_S30.2018-07-12.Fmask.tif
Chipping:  134e 134e.MASK.tif 134e.HLS_S30.stacked.tif 134e.HLS_S30.2018-07-12.Fmask.tif
Chipping:  597b 597b.MASK.tif 597b.HLS_S30.stacked.tif 597b.HLS_S30.2017-07-15.Fmask.tif
Chipping:  623d 623d.MASK.tif 623d.HLS_S30.stacked.tif 623d.HLS_S30.2019-08-19.Fmask.tif
Chipping:  623a 623a.MASK

In [61]:
len(filtered), len(all_chips)

(1, 1544)

In [62]:
chip_paths['output']['hls'].mkdir(exist_ok=True, parents=True)
chip_paths['output']['label'].mkdir(exist_ok=True, parents=True)

for chip in filtered_chips:
    output_data_path = chip_paths['output']['hls'] / chip.data.name 
    output_label_path = chip_paths['output']['label'] / chip.label.name

    shutil.copy2(chip.data, output_data_path)
    shutil.copy2(chip.label, output_label_path)

In [63]:
len(list(chip_paths['output']['hls'].glob('*.tif')))
len(list(chip_paths['output']['label'].glob('*.tif')))

392

In [ ]:
#len(filtered_chips)
#filtered_chips[0]
#CHIPS_PLOTS = PLOT_PATH / 'CHIPS'
#print(len(filtered_chips), len(all_chips))
#for chip in filtered_chips:
#    plot_name = f"{chip.data.name.split('.stacked')[0]}.chip.png"
#    view.view_chip(chip, models.HLS_S30, [2, 1, 0], save_to_file=CHIPS_PLOTS / plot_name, quite=True)

86 308


In [64]:
import numpy as np
import rasterio


def calculate_stats(chip_stacks, n_bands) -> tuple:
    mean = np.zeros(n_bands, dtype=np.float64)
    M2 = np.zeros(n_bands, dtype=np.float64)
    count = np.zeros(n_bands, dtype=np.float64)

    for chip in chip_stacks:
        with rasterio.open(chip.data) as src:
            band_data = src.read()
            count, mean, M2 = 0, 0, 0

            _, H, W = band_data.shape

            batch_count = H * W
            batch_mean = band_data.mean(axis=(1, 2))
            batch_var = band_data.var(axis=(1, 2))

            delta = batch_mean - mean
            total_count = count + batch_count

            mean = mean + delta * (batch_count / total_count)
            M2 = (
                M2
                + batch_var * batch_count
                + (delta**2) * count * batch_count / total_count
            )
            count = total_count

    variance = M2 / count
    std = np.sqrt(variance)

    return mean, std


In [65]:
data_bands

[PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.B.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.G.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.R.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.N.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.SW1.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.SW2.tif')]

In [66]:
chip_means, chip_stds = calculate_stats(filtered_chips, 6)

In [55]:
import json


bands = models.HLS_S30['bands'][:-1]
bands

stats_file = {
    'HLS':{
        'means': {},
        'stds': {},
    }
}

for mean, std, band in zip(chip_means, chip_stds, bands):
    stats_file['HLS']['means'][band.shortname] = float(mean)
    stats_file['HLS']['stds'][band.shortname] = float(std)

stats_file

(output_base / 'statistics.json').write_text(json.dumps(stats_file, indent=2))

415